# Local RAG with Llama 3.2 (Ollama)

This notebook builds a Retrieval-Augmented Generation (RAG) pipeline over the files in `law_data/`.

Important note: RAG is not weight fine-tuning. RAG improves answers by retrieving relevant context at inference time.

You requested local usage with:

```python
OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')
```

In [1]:
# Install once if needed
# %pip install -q openai pypdf numpy

In [10]:
from pathlib import Path
from dataclasses import dataclass
from typing import List, Tuple

import numpy as np
from openai import OpenAI
from pypdf import PdfReader
from IPython.display import display, Markdown

In [3]:
# ---- Config ----
DATA_DIR = Path("../law_data") if Path("../law_data").exists() else Path("law_data")
LLM_MODEL = "llama3.2"
EMBED_MODEL = "nomic-embed-text"  # pull once: ollama pull nomic-embed-text
CHUNK_SIZE = 900
CHUNK_OVERLAP = 150
TOP_K = 4

client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")

print("Data dir:", DATA_DIR.resolve())
print("Using chat model:", LLM_MODEL)
print("Using embedding model:", EMBED_MODEL)

Data dir: C:\Users\Arjun V P\projects\llm_engineering\law_data
Using chat model: llama3.2
Using embedding model: nomic-embed-text


In [4]:
@dataclass
class Chunk:
    source: str
    index: int
    text: str


def read_pdf_text(pdf_path: Path) -> str:
    reader = PdfReader(str(pdf_path))
    pages = []
    for page in reader.pages:
        pages.append(page.extract_text() or "")
    return "\n".join(pages)


def split_text(text: str, chunk_size: int = CHUNK_SIZE, overlap: int = CHUNK_OVERLAP) -> List[str]:
    text = " ".join(text.split())
    chunks = []
    start = 0
    while start < len(text):
        end = min(start + chunk_size, len(text))
        chunks.append(text[start:end])
        if end == len(text):
            break
        start = max(0, end - overlap)
    return chunks


def load_chunks(data_dir: Path) -> List[Chunk]:
    all_chunks: List[Chunk] = []

    for pdf in sorted(data_dir.glob("*.pdf")):
        raw = read_pdf_text(pdf)
        for i, ch in enumerate(split_text(raw)):
            all_chunks.append(Chunk(source=pdf.name, index=i, text=ch))

    for txt in sorted(data_dir.glob("*.txt")):
        raw = txt.read_text(encoding="utf-8", errors="ignore")
        for i, ch in enumerate(split_text(raw)):
            all_chunks.append(Chunk(source=txt.name, index=i, text=ch))

    return all_chunks


def normalize_rows(arr: np.ndarray) -> np.ndarray:
    norms = np.linalg.norm(arr, axis=1, keepdims=True) + 1e-12
    return arr / norms


def embed_with_ollama(texts: List[str], batch_size: int = 32) -> np.ndarray:
    vectors = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i : i + batch_size]
        res = client.embeddings.create(model=EMBED_MODEL, input=batch)
        vectors.extend([d.embedding for d in res.data])
    return normalize_rows(np.array(vectors, dtype=np.float32))


def embed_with_sentence_transformers(texts: List[str]) -> np.ndarray:
    from sentence_transformers import SentenceTransformer

    model = SentenceTransformer("all-MiniLM-L6-v2")
    vecs = model.encode(texts, convert_to_numpy=True, normalize_embeddings=True)
    return np.array(vecs, dtype=np.float32)


EMBEDDING_BACKEND = "ollama"


def embed_texts(texts: List[str], batch_size: int = 32) -> np.ndarray:
    global EMBEDDING_BACKEND
    try:
        EMBEDDING_BACKEND = "ollama"
        return embed_with_ollama(texts, batch_size=batch_size)
    except Exception as exc:
        EMBEDDING_BACKEND = "sentence-transformers"
        print(
            "Falling back to local sentence-transformers embeddings because Ollama embeddings failed:",
            exc,
        )
        return embed_with_sentence_transformers(texts)


def embed_query(query: str) -> np.ndarray:
    if EMBEDDING_BACKEND == "ollama":
        q = client.embeddings.create(model=EMBED_MODEL, input=[query]).data[0].embedding
        qv = np.array(q, dtype=np.float32)
        return qv / (np.linalg.norm(qv) + 1e-12)

    from sentence_transformers import SentenceTransformer

    model = SentenceTransformer("all-MiniLM-L6-v2")
    qv = model.encode([query], convert_to_numpy=True, normalize_embeddings=True)[0]
    return np.array(qv, dtype=np.float32)


chunks = load_chunks(DATA_DIR)
if not chunks:
    raise ValueError(f"No .pdf or .txt files found in {DATA_DIR.resolve()}")

chunk_texts = [c.text for c in chunks]
chunk_embeddings = embed_texts(chunk_texts)

print(f"Loaded {len(chunks)} chunks from {len(set(c.source for c in chunks))} document(s).")
print("Embeddings shape:", chunk_embeddings.shape)
print("Embedding backend:", EMBEDDING_BACKEND)

Loaded 1683 chunks from 3 document(s).
Embeddings shape: (1683, 768)
Embedding backend: ollama


In [5]:
def retrieve(query: str, top_k: int = TOP_K) -> List[Tuple[float, Chunk]]:
    qv = embed_query(query)

    scores = chunk_embeddings @ qv
    idx = np.argsort(scores)[-top_k:][::-1]
    return [(float(scores[i]), chunks[i]) for i in idx]


def answer_with_rag(query: str, top_k: int = TOP_K) -> str:
    retrieved = retrieve(query, top_k=top_k)

    context_blocks = []
    for score, ch in retrieved:
        context_blocks.append(
            f"[source={ch.source} chunk={ch.index} score={score:.4f}]\n{ch.text}"
        )

    context = "\n\n".join(context_blocks)

    system_prompt = (
        "You are a legal assistant. Use only the provided context when possible. "
        "If context is insufficient, clearly say what is missing."
    )

    user_prompt = (
        f"Question:\n{query}\n\n"
        f"Retrieved context:\n{context}\n\n"
        "Answer with concise bullet points and cite source chunk tags like [source=... chunk=...]."
    )

    response = client.chat.completions.create(
        model=LLM_MODEL,
        temperature=0.1,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
    )
    return response.choices[0].message.content


query = "What are the Fundamental Rights described in this constitution text?"
print(answer_with_rag(query, top_k=4))

Unfortunately, the provided text does not mention Fundamental Rights explicitly. However, some articles related to individual rights can be inferred:

* Article 300A: "No person shall be deprived of his property save by authority of law." [source=constitution_english.pdf chunk=568]
* No other specific Fundamental Rights are mentioned in the provided chunks.

If you'd like to know more about Fundamental Rights in general, I can provide information on the Indian Constitution's Part III (Fundamental Rights) or help with a different question.


In [6]:
# Try your own question:
print(answer_with_rag("Explain the right to equality.", top_k=5))

Here is a concise explanation of the right to equality:

• The right to equality is a fundamental human right that guarantees equal treatment and opportunities for all individuals.
• It is a principle of non-discrimination, ensuring that no one is discriminated against on grounds such as sex, caste, ethnicity, place of birth, disability, or social background. [source=jjact2015.pdf chunk=28]
• The principle of equality is also linked to the right to privacy and confidentiality, ensuring that individuals have the right to protection of their personal information. [source=ChildRightsandProtection-English (Final).pdf.pdf chunk=57]
• Equality is essential for promoting dignity and security, particularly for marginalized and oppressed communities.
• Human rights standards and international conventions, such as the UNCRC, recognize the importance of equality and non-discrimination in protecting human rights. [source=ChildRightsandProtection-English (Final).pdf.pdf chunk=39]
• The right to equ

In [12]:
print(display(Markdown(answer_with_rag("Explain laws on children.", top_k=5))))

Here are the laws on children in India:

**Definitions of Children:**

* Right to Education Act, 2009: Children aged 6-14 years (chunk=77 score=0.7750)
* Child Labor (Prohibition and Regulation) Act, 1986: Children aged 14 years (chunk=86 score=0.7662)
* Mines Act 1952: Children aged 18 years (chunk=77 score=0.7750)
* Factories Act 1948: Children aged 14 years (chunk=77 score=0.7750)

**Juvenile Justice Laws:**

* Juvenile Justice Act, 1986: Defines a child as a person who has not completed 18 years of age (chunk=162 score=0.7531)
* Juvenile Justice (Care and Protection of Children) Act, 2000: Defines a child as a person who has not completed 18 years of age (chunk=85 score=0.7412)

**Constitutional Rights:**

* Right to Equality: Article 14 (source=ChildRightsandProtection-English (Final).pdf.pdf chunk=169 score=0.7541)
* Right to Freedom: Article 19 (source=ChildRightsandProtection-English (Final).pdf.pdf chunk=169 score=0.7541)
* Right against Exploitation: Article 23 (source=ChildRightsandProtection-English (Final).pdf.pdf chunk=169 score=0.7541)
* Right to Education: Article 21A (source=ChildRightsandProtection-English (Final).pdf.pdf chunk=169 score=0.7541)
* Right to Protection: Article 24 (source=ChildRightsandProtection-English (Final).pdf.pdf chunk=85 score=0.7412)
* Right to Health and Nutrition: Article 39 (source=ChildRightsandProtection-English (Final).pdf.pdf chunk=169 score=0.7541)
* Right to Protection of Cultural and Educational Rights: Article 29 (source=ChildRightsandProtection-English (Final).pdf.pdf chunk=85 score=0.7412)

**Policies and Institutions:**

* National Policy for Children (1974) (source=ChildRightsandProtection-English (Final).pdf.pdf chunk=169 score=0.7541)
* Integrated Child Development Services (ICDS) (1975) (source=ChildRightsandProtection-English (Final).pdf.pdf chunk=85 score=0.7412)
* Tamil Nadu Integrated Nutrition Project (TINP) (1980s) (source=ChildRightsandProtection-English (Final).pdf.pdf chunk=169 score=0.7541)

Note: The provided context is limited, and some information may be missing or not explicitly stated in the source chunks.

None
